# CAP PME : contrôle et préparation des données de la liasse fiscale

In [28]:
import pandas as pd

## Chargement du référentiel de la liasse fiscale

In [29]:
# Import du référentiel
fichier_referentiel ="https://raw.githubusercontent.com/Hines98/CAP-PME/refs/heads/main/Referentiel_liasse.csv"
ref = pd.read_csv(fichier_referentiel, sep =';', encoding='utf-8-sig')
ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code               138 non-null    str    
 1   intitule_officiel  138 non-null    str    
 2   agregat            138 non-null    str    
 3   signe_agregat      138 non-null    int64  
 4   sens_resultat      41 non-null     float64
 5   statut             138 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 6.6 KB


In [30]:
# Création d'un dictionnaire
dico = dict(zip(ref['code'], ref['intitule_officiel']))
len(dico)

138

In [31]:
dico

{'AB': "Frais d'établissement",
 'AC': "Frais d'établissement",
 'CX': 'Frais de développement',
 'CQ': 'Frais de développement',
 'AF': 'Concessions, brevets et droits similaires',
 'AG': 'Concessions, brevets et droits similaires',
 'AH': 'Fonds commercial',
 'AI': 'Fonds commercial',
 'AJ': 'Autres immobilisations incorporelles',
 'AK': 'Autres immobilisations incorporelles',
 'AL': 'Immobilisations incorporelles en cours, avances et acomptes',
 'AM': 'Immobilisations incorporelles en cours, avances et acomptes',
 'AN': 'Terrains',
 'AO': 'Terrains',
 'AP': 'Constructions',
 'AQ': 'Constructions',
 'AR': 'Installations techniques, matériel et outillage industriels',
 'AS': 'Installations techniques, matériel et outillage industriels',
 'AT': 'Autres immobilisations corporelles',
 'AU': 'Autres immobilisations corporelles',
 'AV': 'Immobilisations corporelles en cours, avances et acomptes',
 'AW': 'Immobilisations corporelles en cours, avances et acomptes',
 'CS': 'Participations',
 

## Chargement des Etats financiers de l'entreprise

In [32]:
# Import du fichier
fichier_etats = "https://raw.githubusercontent.com/Hines98/CAP-PME/main/Fichier_type_etats_financiers.xlsx"
etats = pd.read_excel(fichier_etats, sheet_name=None)
etats.keys()

dict_keys(['Bilan', 'Compte de résultat', 'Annexe'])

In [33]:
# Concaténation des feuilles
data = pd.concat(etats.values(), ignore_index=True)
data.shape

(138, 5)

In [34]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    138 non-null    str    
 1   Poste   138 non-null    str    
 2   2023    50 non-null     float64
 3   2024    50 non-null     float64
 4   2025    50 non-null     float64
dtypes: float64(3), str(2)
memory usage: 5.5 KB


In [35]:
# Reconnaissance des codes des Etats financiers dans le dictionnaire de référence
inconnus = data[~data['Code'].isin(dico)]
inconnus

,Code,Poste,2023,2024,2025


In [36]:
if len(inconnus) > 0:
    raise ValueError(f"Codes non reconnus : {list(inconnus['Code'])}. Corrigez le fichier puis relancez.")
else:
    print("Tous les codes sont reconnus.")

Tous les codes sont reconnus.


In [37]:
# Remplacement des lignes vides par 0
annees = data.columns[2:]
data[annees] = data[annees].fillna(0)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    138 non-null    str    
 1   Poste   138 non-null    str    
 2   2023    138 non-null    float64
 3   2024    138 non-null    float64
 4   2025    138 non-null    float64
dtypes: float64(3), str(2)
memory usage: 5.5 KB


In [38]:
# Avertissement sur le nombre d'années d'exercices
nb_annees = len(annees)
if nb_annees < 3:
    print(f"Attention : {nb_annees} exercice(s) renseigné(s). Trois exercices sont recommandés pour des repères historiques fiables.")
else:
    print(f"{nb_annees} exercices renseignés.")

3 exercices renseignés.


In [39]:
# Ajout de nouvelles colonnes aux Etats financiers
data_new = data.merge(ref, left_on='Code', right_on='code').drop(columns='code')
data_new.shape

(138, 10)

In [40]:
data_new.head()

,Code,Poste,2023,2024,2025,intitule_officiel,agregat,signe_agregat,sens_resultat,statut
0,AA,Capital souscrit non appelé,0.0,0.0,0.0,Capital souscrit non appelé,Autres actifs,1,NaN,Facultatif
1,AB,Frais d'établissement – brut,0.0,0.0,0.0,Frais d'établissement,Actif immobilisé,1,NaN,Facultatif
2,AC,Frais d'établissement – amortissements et dépr...,0.0,0.0,0.0,Frais d'établissement,Actif immobilisé,-1,NaN,Facultatif
3,CX,Frais de développement – brut,0.0,0.0,0.0,Frais de développement,Actif immobilisé,1,NaN,Facultatif
4,CQ,Frais de développement – amortissements et dép...,0.0,0.0,0.0,Frais de développement,Actif immobilisé,-1,NaN,Facultatif


## Contrôles indispensables

In [41]:
# Identification de poste indispensables à 0
indispensables = data_new[data_new['statut']=='Indispensable']
a_zero = indispensables[(indispensables[annees]==0).any(axis=1)]
a_zero[['Code', 'Poste', 'statut']]

,Code,Poste,statut


In [42]:
# Avertissement si postes indispensables à 0
if len(a_zero) > 0:
    print("Attention : postes indispensables à zéro, à vérifier :")
    print(a_zero[["Code", "Poste"] + list(annees)])
else:
    print("Aucun poste indispensable à zéro.")

Aucun poste indispensable à zéro.


In [43]:
# Vérification du chiffre d'affaires total doit être positif chaque année
ca = data_new[data_new["agregat"] == "Chiffre d'affaires"][annees].sum()
ca

2023    10300000.0
2024    11100000.0
2025    12000000.0
dtype: float64

In [44]:
if (ca < 0).any():
    raise ValueError("Chiffre d'affaires négatif sur au moins une année. Corrigez le fichier puis relancez.")
elif (ca == 0).any():
    print("Attention : chiffre d'affaires nul sur au moins une année, à vérifier.")
else:
    print("Chiffre d'affaires renseigné sur toutes les années.")

Chiffre d'affaires renseigné sur toutes les années.


In [45]:
# Vérification du bilan équilibré (actif net = passif)
actif = ["Actif immobilisé", "Stocks", "Créances clients",
         "Autres actifs circulants", "Trésorerie", "Autres actifs"]
passif = ["Capitaux propres", "Autres fonds propres", "Provisions",
          "Dettes financières", "Dettes fournisseurs", "Dettes fiscales et sociales",
          "Autres dettes à court terme", "Autres passifs"]

montants_signes = data_new[annees].multiply(data_new["signe_agregat"], axis=0)

total_actif = montants_signes[data_new["agregat"].isin(actif)].sum()
total_passif = montants_signes[data_new["agregat"].isin(passif)].sum()

ecart = total_actif - total_passif
ecart

2023    0.0
2024    0.0
2025    0.0
dtype: float64

In [46]:
# Avertissement en cas de déséquilibre actif et passif du bilan
bilan = pd.DataFrame({"Total actif": total_actif,
                      "Total passif": total_passif,
                      "Écart": ecart})

desequilibre = bilan[bilan["Écart"].abs() > 1]

if len(desequilibre) > 0:
    print("Bilan déséquilibré sur les années suivantes :")
    print(desequilibre)
    raise ValueError("Corrigez le fichier puis relancez.")
else:
    print("Bilan équilibré sur toutes les années.")

Bilan équilibré sur toutes les années.


In [47]:
# Vérification de l'égalité résultat = bilan
resultat_calcule = data_new[annees].multiply(data_new["sens_resultat"], axis=0).sum()
resultat_bilan = data_new[data_new["Code"] == "DI"][annees].sum()

resultat = pd.DataFrame({"Résultat recalculé": resultat_calcule,
                         "Résultat du bilan (DI)": resultat_bilan,
                         "Écart": resultat_calcule - resultat_bilan})
resultat

,Résultat recalculé,Résultat du bilan (DI),Écart
2023,122000.0,122000.0,0.0
2024,176000.0,176000.0,0.0
2025,277000.0,277000.0,0.0


In [48]:
# Avertissement en cas d'écart entre résultat et bilan
incoherent = resultat[resultat["Écart"].abs() > 1]

if len(incoherent) > 0:
    print("Résultat incohérent sur les années suivantes :")
    print(incoherent)
    raise ValueError("Corrigez le fichier puis relancez.")
else:
    print("Résultat cohérent sur toutes les années.")

Résultat cohérent sur toutes les années.


## Création de la table d'alimentation du power BI

In [49]:
# Totaux par agrégat
totaux_bilan = montants_signes.groupby(data_new["agregat"]).sum()

montants_sens = data_new[annees].multiply(data_new["sens_resultat"], axis=0)
totaux_resultat = montants_sens.groupby(data_new["agregat"]).sum()

In [50]:
# Soldes du compte de résultat ==> chaque liste reprend la précédente et ajoute des agrégats
va = ["Chiffre d'affaires", "Production stockée et immobilisée",
      "Coût des marchandises vendues", "Matières consommées", "Charges externes"]

ebe = va + ["Subventions d'exploitation", "Impôts et taxes", "Charges de personnel"]

rex = ebe + ["Reprises (exclues de la CAF)", "Autres produits et charges d'exploitation",
             "Dotations (réintégrées dans la CAF)"]

rcai = rex + ["Opérations en commun", "Produits financiers", "Reprises financières",
              "Dotations financières", "Intérêts de la dette", "Autres charges financières"]

charges_exploitation = ["Coût des marchandises vendues", "Matières consommées",
                        "Charges externes", "Impôts et taxes", "Charges de personnel"]

compte_resultat = pd.DataFrame({
    "Chiffre d’affaires": totaux_resultat.loc["Chiffre d'affaires"],
    "Valeur ajoutée": totaux_resultat.loc[va].sum(),
    "Total charges d’exploitation (hors amort.)": -totaux_resultat.loc[charges_exploitation].sum(),
    "Excédent Brut d’Exploitation (EBE)": totaux_resultat.loc[ebe].sum(),
    "Dotations aux amortissements": -totaux_resultat.loc[["Dotations (réintégrées dans la CAF)", "Dotations financières"]].sum(),
    "Reprises": totaux_resultat.loc[["Reprises (exclues de la CAF)", "Reprises financières"]].sum(),
    "Résultat d’exploitation": totaux_resultat.loc[rex].sum(),
    "Charges financières": -totaux_resultat.loc["Intérêts de la dette"],
    "Résultat courant avant impôt": totaux_resultat.loc[rcai].sum(),
    "Impôt sur les sociétés": -totaux_resultat.loc["Impôt sur les sociétés"],
    "Résultat net": totaux_resultat.sum()})
compte_resultat

,Chiffre d’affaires,Valeur ajoutée,Total charges d’exploitation (hors amort.),Excédent Brut d’Exploitation (EBE),Dotations aux amortissements,Reprises,Résultat d’exploitation,Charges financières,Résultat courant avant impôt,Impôt sur les sociétés,Résultat net
2023,10300000.0,3447000.0,9612000.0,716000.0,470000.0,20000.0,256000.0,68000.0,191000.0,41000.0,122000.0
2024,11100000.0,3818000.0,10296000.0,849000.0,520000.0,25000.0,341000.0,74000.0,270000.0,58000.0,176000.0
2025,12000000.0,4179000.0,11020000.0,1027000.0,560000.0,35000.0,490000.0,73000.0,421000.0,92000.0,277000.0


In [51]:
# Lignes du bilan et de l'annexe
lignes_bilan = pd.DataFrame({
    "Immobilisations nettes": totaux_bilan.loc["Actif immobilisé"],
    "Stocks": totaux_bilan.loc["Stocks"],
    "Créances clients": totaux_bilan.loc["Créances clients"],
    "Autres actifs circulants": totaux_bilan.loc["Autres actifs circulants"],
    "Trésorerie": totaux_bilan.loc["Trésorerie"],
    "Capitaux propres": totaux_bilan.loc["Capitaux propres"],
    "Dettes financières": totaux_bilan.loc["Dettes financières"],
    "Dettes fournisseurs": totaux_bilan.loc["Dettes fournisseurs"],
    "Dettes à moins d'un an": totaux_bilan.loc["Dettes à moins d'un an"],
    "Emprunts remboursés": totaux_bilan.loc["Remboursement de la dette existante"],
    "Concours bancaires courants": totaux_bilan.loc["Concours bancaires courants"],
    "Investissements": totaux_bilan.loc["Investissements"]})
lignes_bilan

,Immobilisations nettes,Stocks,Créances clients,Autres actifs circulants,Trésorerie,Capitaux propres,Dettes financières,Dettes fournisseurs,Dettes à moins d'un an,Emprunts remboursés,Concours bancaires courants,Investissements
2023,3250000.0,1128000.0,1429000.0,226000.0,869000.0,2852000.0,2100000.0,1057000.0,2280000.0,400000.0,0.0,480000.0
2024,3865000.0,1216000.0,1540000.0,244000.0,752000.0,2985000.0,2520000.0,1133000.0,2482000.0,430000.0,0.0,1100000.0
2025,3780000.0,1315000.0,1665000.0,264000.0,472000.0,3200000.0,2040000.0,1217000.0,2646000.0,480000.0,0.0,450000.0


In [52]:
# Table finale pour Power BI
table = pd.concat([compte_resultat, lignes_bilan], axis=1)
table.index.name = "Année"

table.to_excel("Etats_financiers_power_bi.xlsx")

table.shape

(3, 23)